# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a reproducible workflow to load and explore the FAIRˆ² dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is defined by a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Date published: {metadata.date_published}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s. This allows us to know the data structure and what can be loaded.

We will enumerate all registered record sets and, for each, show their fields (columns) and their associated `@id`.

In [ ]:
# Utility to inspect structure
def print_record_sets_and_fields(ds):
    print(f"Dataset Record Sets (@id):")
    for rs in ds.record_sets:
        print(f"- RecordSet Name: {rs.name}, @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print(f"  Fields for {rs.name} (@id):")
            for field in rs.fields:
                field_type = getattr(field, 'data_type', '')
                print(f"    - {field.name}, @id: {field.id} [{field_type}]")
        elif hasattr(rs, 'columns') and rs.columns:
            print(f"  Columns for {rs.name} (@id):")
            for col in rs.columns:
                col_type = getattr(col, 'data_type', '')
                print(f"    - {col.name}, @id: {col.id} [{col_type}]")
        else:
            print("  (No fields/columns defined)")

# Print record sets and fields by @id
print_record_sets_and_fields(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**All entities (record sets, fields, columns) are referenced by their `@id` as required.**

Below, we'll select all available record sets, read their data, and place each into a DataFrame keyed by the record set `@id`.

In [ ]:
# List all record set @id
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record set IDs:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Use first record set with data for demonstration
main_rs_id = None
for rsid in record_set_ids:
    if rsid in dataframes:
        main_rs_id = rsid
        break

if main_rs_id:
    print(f"\nDataFrame columns in record set '@id': {main_rs_id}\n", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No data frames loaded. Please check the dataset schema or data availability.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing and transformation steps. 

* We will choose a numeric field (by @id), filter records, normalize values, and group by another field (by @id) if available.

**NOTE**: The actual `@id` values for fields must be substituted; below, we auto-detect possible numeric and categorical fields.

In [ ]:
import numpy as np

# Identify numeric columns (by @id)
if main_rs_id is not None:
    df = dataframes[main_rs_id]
    # Try to infer numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric columns (potential field @id):", numeric_cols)
    # Use the first numeric column
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Guess a suitable group/categorical field (by @id)
        non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
        print(f"Categorical/text columns (potential group field @id): {non_numeric}")

        group_field = None
        # Use first categorical with <50 unique values for grouping
        for col in non_numeric:
            if df[col].nunique() < 50:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean')
            print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable group field detected.")
    else:
        print("No numeric fields available in the main DataFrame.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the selected record set.

For example, we plot the distribution of the selected numeric field, and if a grouping field is available, compare means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and numeric_cols:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field is defined, boxplot by group
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No plot: insufficient data.')

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load and inspect dataset metadata and structure from a Croissant schema using `mlcroissant`
- List available record sets and fields referenced by their `@id`
- Extract record set data into pandas DataFrames for analysis
- Apply basic EDA: filtering, normalization, grouping, and visualization

This workflow can be reused for other Croissant-structured datasets and extended for deeper analysis, such as regression, advanced visualizations, or export for use in ML pipelines.